In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv')

/tmp/ipykernel_16/3068129054.py:1: DtypeWarning: Columns (38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv('/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv')


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138701 entries, 0 to 138700
Data columns (total 50 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   TransactionID              138701 non-null  int64  
 1   TargetValue                138701 non-null  float64
 2   AssetID                    138701 non-null  int64  
 3   ProductConfigID            138701 non-null  int64  
 4   DataOriginCode             138701 non-null  object 
 5   VendorPartnerID            138701 non-null  object 
 6   ManufactureYear            138701 non-null  int64  
 7   OperationalHoursMeter      82058 non-null   float64
 8   UtilizationTier            50475 non-null   object 
 9   TransactionDate            138701 non-null  object 
 10  Spec_FullDescriptor        138701 non-null  object 
 11  Spec_BaseClass             138701 non-null  object 
 12  Spec_SubClass              101716 non-null  object 
 13  Spec_ReleaseSeries         32

In [4]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 49 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   TransactionID              15000 non-null  int64  
 1   AssetID                    15000 non-null  int64  
 2   ProductConfigID            15000 non-null  int64  
 3   DataOriginCode             15000 non-null  object 
 4   VendorPartnerID            15000 non-null  object 
 5   ManufactureYear            15000 non-null  int64  
 6   OperationalHoursMeter      8809 non-null   float64
 7   UtilizationTier            5356 non-null   object 
 8   TransactionDate            15000 non-null  object 
 9   Spec_FullDescriptor        15000 non-null  object 
 10  Spec_BaseClass             15000 non-null  object 
 11  Spec_SubClass              11006 non-null  object 
 12  Spec_ReleaseSeries         3505 non-null   object 
 13  Spec_VariantModifier       5609 non-null   obj

In [5]:
train_df.head()

,TransactionID,TargetValue,AssetID,ProductConfigID,DataOriginCode,VendorPartnerID,ManufactureYear,OperationalHoursMeter,UtilizationTier,TransactionDate,...,col20,col21,col22,col23,col24,col25,col27,col28,col29,col30
0,1139309,57000.0,117665,88,ACH138,GTX,1997,4640.0,Low,2005-03-26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standard,Conventional
1,1139316,26500.0,1001282,4616,ACH138,GTX,2005,508.0,Low,2009-12-18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1139317,21000.0,772709,1948,ACH138,GTX,1994,11540.0,High,2005-08-26,...,Steel,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN
3,1139322,27000.0,902010,3550,ACH138,GTX,2002,4883.0,High,2006-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1139333,21500.0,1036259,36014,ACH138,GTX,2009,302.0,Low,2010-08-27,...,Rubber,None or Unspecified,None or Unspecified,None or Unspecified,None or Unspecified,Double,NaN,NaN,NaN,NaN


In [6]:
target_variable = set(train_df.columns) - set(test_df.columns)

print(f'The target variable is : {target_variable}')

The target variable is : {'TargetValue'}


In [7]:
numerical_cols = train_df.select_dtypes(include = ['int64','float64']).columns

num_count = len(numerical_cols)

print(f'Number of numerical columns are : {num_count}')

Number of numerical columns are : 6


In [8]:
missing_count = train_df.isnull().sum()

high_missing = missing_count[missing_count > 124800]

print(f'Columns with high missing values are : {high_missing}')

Columns with high missing values are : col4     128320
col5     128320
col18    138635
col19    138635
dtype: int64


In [9]:
OHM_len = len(train_df['OperationalHoursMeter'])

OHM_len_missing = train_df['OperationalHoursMeter'].isnull().sum()

missing_percentage = (OHM_len_missing / OHM_len) * 100

print(f"Missing values in OperationalHoursMeter : {missing_percentage:.2f}%")

Missing values in OperationalHoursMeter : 40.84%


In [10]:
Target_Median = train_df['TargetValue'].median()
print(f'The Median is : {Target_Median} ')

The Median is : 35000.0 


In [11]:
Mode = train_df['ManufactureYear'].mode()[0]
print(f'The Mode of ManufactureYear is : {Mode}')

The Mode of ManufactureYear is : 1001


In [12]:
Min_date = train_df['TransactionDate'].min()
print(f'The Minimum date in TransactionDate is : {Min_date}')

The Minimum date in TransactionDate is : 1990-01-31


In [13]:
Unique_count = train_df['Spec_BaseClass'].nunique()
print(f'Number of unique  string classes exist within the Spec_BaseClass column is : {Unique_count}')

Number of unique  string classes exist within the Spec_BaseClass column is : 1249


In [14]:
Highest_vol = train_df['RegionCode'].mode()[0]
print(f'Region accounts for the highest volume of machinery transactions is : {Highest_vol}')

Region accounts for the highest volume of machinery transactions is : Florida


In [15]:
Florida_Machine = train_df[train_df['RegionCode'] == 'Florida']

Avg_Value = Florida_Machine['TargetValue'].mean()
print(f'The average TargetValue for machines sold is : {Avg_Value:.2f}')


The average TargetValue for machines sold is : 45007.54


In [16]:
Common_Tier = train_df['UtilizationTier'].mode()[0]
print(f'Most common tier is : {Common_Tier}')

Most common tier is : Medium


In [17]:
Correlation = train_df['TargetValue'].corr(train_df['OperationalHoursMeter'])
print(f'The correlation coefficient is : {Correlation}')

The correlation coefficient is : 0.0019936232351365833


In [18]:
Most_Common_Machinery = train_df['FunctionalClassification'].mode()[0]
print(f'The most common machinery is : {Most_Common_Machinery}')


The most common machinery is : Hydraulic Excavator, Track - 21.0 to 24.0 Metric Tons


In [19]:
#Replace 1001 with NaN
train_df['ManufactureYear'] = train_df['ManufactureYear'].replace(1001,np.nan)

In [20]:
#1 Convert TransactionDate to a proper datetime object
train_df['TransactionDate'] = pd.to_datetime(train_df['TransactionDate'])

# 2. Extract Year and Quarter
train_df['TransactionYear'] = train_df['TransactionDate'].dt.year
train_df['TransactionQuarter'] = train_df['TransactionDate'].dt.quarter

# 3. Calculate Asset Age
train_df['AssetAge'] = train_df['TransactionYear'] - train_df['ManufactureYear']

# 4. Calculate Descriptor Length
train_df['DescriptorLength'] = train_df['Spec_FullDescriptor'].str.len()

# 5. Create Binary Flags
train_df['HasOperationalHours'] = train_df['OperationalHoursMeter'].notna().astype(int)
train_df['HasVariantModifier'] = train_df['Spec_VariantModifier'].notna().astype(int)

In [21]:
#checking all the changes
new_cols = ['ManufactureYear', 'TransactionYear', 'TransactionQuarter', 
            'AssetAge', 'DescriptorLength', 'HasOperationalHours', 'HasVariantModifier']
display(train_df[new_cols].head(10))

,ManufactureYear,TransactionYear,TransactionQuarter,AssetAge,DescriptorLength,HasOperationalHours,HasVariantModifier
0,1997.0,2005,1,8.0,6,1,0
1,2005.0,2009,4,4.0,4,1,0
2,1994.0,2005,3,11.0,6,1,1
3,2002.0,2006,4,4.0,4,1,0
4,2009.0,2010,3,1.0,6,1,0
5,NaN,2008,3,NaN,4,1,0
6,2005.0,2009,3,4.0,4,1,0
7,1999.0,2006,4,7.0,8,1,1
8,2002.0,2007,1,5.0,4,1,0
9,2005.0,2007,1,2.0,4,1,0


In [22]:
Freq_Assest_Age = train_df['AssetAge'].mode()[0]
print(f'The most frequent assest age is : {Freq_Assest_Age}')

The most frequent assest age is : 5.0


In [23]:
quarterly_averages = train_df.groupby('TransactionQuarter')['TargetValue'].mean().sort_values(ascending=False)

top_quarter = quarterly_averages.index[0]
print(f'The quarter with highest avg is : {top_quarter}')

The quarter with highest avg is : 1


In [24]:
Max_descriptor_len = train_df['DescriptorLength'].max()
print(f'Max descriptor length is : {Max_descriptor_len}')

Max descriptor length is : 19


In [25]:
hours_count = (train_df['HasOperationalHours'] == 1).sum()
print(f"Records with HasOperationalHours = 1 : {hours_count}")

Records with HasOperationalHours = 1 : 82058


In [26]:
variant_percentage = train_df['HasVariantModifier'].mean() * 100
print(f'percentage of records contain variant information is : {variant_percentage:.2f}')

percentage of records contain variant information is : 37.73


In [27]:
ManufactureYear_mean = train_df['ManufactureYear'].mean()
print(f'The mean of  ManufactureYear is : {ManufactureYear_mean:.2f}')

The mean of  ManufactureYear is : 1997.38
